# Active Learning Quickstart

This notebook just quickly goes through `active_learning` package with a toy setup. Mainly so that I remember how to use it once we decide on a base learner.

Real project workflow will look the same:

1. Build a candidate table for the pool.
2. Plug in model adapters for the trained base models.
3. Create scorers: uncertainty and ERM-vs-REx disagreement.
4. Use the engine to rank the unlabeled pool and query the next point or batch.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Add the package root to sys.path so `import active_learning` works whether
# the notebook is launched from the project root or from the notebooks folder.
possible_roots = [
    Path.cwd() / "Inv-SHAF" / "active-learning",
    Path.cwd().parent,
    Path.cwd(),
]

for root in possible_roots:
    if (root / "active_learning").exists():
        sys.path.insert(0, str(root))
        break
else:
    raise RuntimeError("Could not locate the active_learning package.")

from active_learning import (
    ActiveLearningEngine,
    CombinedScorer,
    DisagreementScorer,
    PoolManager,
    TopKSelector,
    UncertaintyScorer,
)


## 1. Build a candidate table

Each row represents one candidate in the pool. In the real project, this table would likely contain fields such as:

- `spot_id`
- `donor_id`
- `patch_path`
- `sample_id`
- any metadata needed later for analysis or constraints


In [ ]:
candidates = pd.DataFrame(
    {
        "spot_id": [f"spot_{i}" for i in range(8)],
        "donor_id": ["D1", "D1", "D2", "D2", "D3", "D3", "D4", "D4"],
        "patch_path": [f"patches/spot_{i}.png" for i in range(8)],
        # `feature` is just a toy stand-in for whatever the future model will use.
        "feature": [0.15, 0.30, 0.20, 0.80, 0.40, 0.65, 0.55, 0.95],
    }
)

candidates


## 2. Create model adapters

The active learning package does **not** care what the real base model architecture is. It only expects a small adapter with methods like:

- `predict(candidates)`
- `predict_uncertainty(candidates)`

Below, `ToyModel` stands in for a future trained ERM model or REx model.

Tried to make it as agnostic as possible.

In [ ]:
class ToyModel:
    """Small deterministic adapter used only for demonstration."""

    def __init__(self, weight: float, bias: float, uncertainty_scale: float = 1.0):
        self.weight = weight
        self.bias = bias
        self.uncertainty_scale = uncertainty_scale

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        # In the real project this would call the trained model and return one
        # prediction vector per candidate.
        feature = frame["feature"].to_numpy(dtype=float)
        return np.stack(
            [
                self.weight * feature + self.bias,
                0.5 * self.weight * feature - self.bias,
            ],
            axis=1,
        )

    def predict_uncertainty(self, frame: pd.DataFrame) -> np.ndarray:
        # In the real project this might come from MC-dropout, ensembles, or
        # another uncertainty method chosen by the team.
        feature = frame["feature"].to_numpy(dtype=float)
        return np.abs(feature - 0.5) * self.uncertainty_scale


erm_model = ToyModel(weight=1.00, bias=0.00, uncertainty_scale=1.00)
rex_model = ToyModel(weight=0.65, bias=0.10, uncertainty_scale=0.75)


## 3. Assemble the pool manager, scorers, and engine

Here we treat one spot as already labeled just so the example looks like a real active learning round.


In [ ]:
pool = PoolManager(
    candidates=candidates,
    id_column="spot_id",
    initial_labeled_ids=["spot_0"],
)

scorer = CombinedScorer(
    scorers={
        "uncertainty": UncertaintyScorer(erm_model),
        "invariance": DisagreementScorer(erm_model, rex_model),
    },
    weights={
        "uncertainty": 0.60,
        "invariance": 0.40,
    },
)

engine = ActiveLearningEngine(
    pool_manager=pool,
    scorer=scorer,
    selector=TopKSelector(),
)


## 4. Query the next point

The engine ranks the unlabeled pool and returns both:

- the fully scored unlabeled pool
- the selected point or batch


In [ ]:
result = engine.query(k=1)

print("Selected candidate(s):")
display(result.selected[["spot_id", "donor_id", "score", "uncertainty.uncertainty", "invariance.disagreement"]])

print("Top ranked unlabeled candidates:")
display(
    result.scored_pool.sort_values("score", ascending=False).head(5)[
        ["spot_id", "donor_id", "score", "uncertainty.uncertainty", "invariance.disagreement"]
    ]
)


## 5. Commit the query

This simulates what happens after you request the next label and move that point into the labeled set.


In [ ]:
revealed = engine.commit(result.selected)

print("New labeled pool size:", len(pool.get_labeled()))
print("Remaining unlabeled pool size:", len(pool.get_unlabeled()))
display(revealed[["spot_id", "donor_id", "feature"]])


## 6. How to plug in the real models later

When you finish the model Shreyan, you just need to replace `ToyModel` a few teeny adapters around the real trained models.

Example:

- `erm_model.predict(frame)` should return gene predictions from the ERM model.
- `erm_model.predict_uncertainty(frame)` should return one uncertainty value per candidate.
- `rex_model.predict(frame)` should return gene predictions from the robust model.

Everything else in the notebook can stay the same.
